In [19]:
import pandas as pd
import numpy as np
from scipy import stats
import os


In [20]:
data_dir = '../data/'

In [21]:
df = pd.read_csv(os.path.join(data_dir, 'combined_healthy_disease.csv'))
df.head()

,Sample_ID,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,...,ENSG00000283787,ENSG00000283907,ENSG00000283913,ENSG00000284032,ENSG00000284373,ENSG00000284387,ENSG00000284395,ENSG00000284505,ENSG00000284552,Disease
0,GSM1501013,8.002176,0.175238,4.949994,3.285351,0.959516,0.672894,59.325067,4.168056,8.104327,...,-0.246719,3.031789,0.903138,1.222211,0.132392,0.491394,-0.079424,-0.255411,3.071849,Healthy Control
1,GSM1501015,10.557890,0.097413,7.680354,2.795125,0.789783,0.733455,43.835288,5.133417,9.559194,...,-0.248596,1.914932,0.424076,0.423539,-0.155705,0.086307,0.230390,-0.489850,1.632879,Healthy Control
2,GSM1501016,10.272135,0.066298,8.835539,3.166677,0.941032,0.755181,51.390227,4.168056,10.980774,...,-0.086860,1.313128,0.501803,0.357365,-0.321937,0.232894,-0.066718,-0.233382,2.117271,Healthy Control
3,GSM1501017,8.290414,-0.039647,5.792685,2.203175,0.846344,0.612263,55.423440,3.657327,10.710821,...,0.337941,4.507792,0.120017,0.452607,-0.263615,0.443599,0.291308,-0.474034,3.032463,Healthy Control
4,GSM1501019,10.677884,0.148528,4.299517,3.382250,0.549977,0.720861,52.831945,2.984661,7.609664,...,-0.195830,2.945203,0.872271,0.371177,-0.042413,-0.008590,0.158892,-0.567165,3.264821,Fatty Liver Disease


In [22]:

df_normal = df[df['Disease'] == 'Healthy Control']
df_tumor = df[df['Disease'] == 'Fatty Liver Disease']

df_normal = df_normal.drop(columns=['Sample_ID', 'Disease'])
df_tumor = df_tumor.drop(columns=['Sample_ID', 'Disease'])

genes = []
log2_fold_changes = []
p_values = []

gene_names = df_normal.columns

for gene in gene_names:
    normal_expression = df_normal[gene].values
    tumor_expression = df_tumor[gene].values
    
    mean_normal = np.mean(normal_expression)
    mean_tumor = np.mean(tumor_expression)
    
    # Add a small value to avoid division by zero
    if mean_normal == 0:
        mean_normal = 1e-10
    if mean_tumor == 0:
        mean_tumor = 1e-10

    log2fc = np.log2(mean_tumor / mean_normal)
    
    t_stat, p_value = stats.ttest_ind(tumor_expression, normal_expression, nan_policy='omit', equal_var=False)

    genes.append(gene)
    log2_fold_changes.append(log2fc)
    p_values.append(p_value)

results_df = pd.DataFrame({
    'gene': genes,
    'log2FC': log2_fold_changes,
    'p_value': p_values
})


/var/folders/_k/0mgp9cxj72dcxh4819b8hd740000gn/T/ipykernel_4086/2008194121.py:26: RuntimeWarning: invalid value encountered in log2
  log2fc = np.log2(mean_tumor / mean_normal)


In [23]:

results_df.to_csv(os.path.join(data_dir, 'differential_expression_results.csv'), index=False)

results_df.head()

,gene,log2FC,p_value
0,ENSG00000000003,0.079494,0.108098
1,ENSG00000000005,0.550160,0.369121
2,ENSG00000000419,0.059139,0.413201
3,ENSG00000000457,-0.107665,0.009359
4,ENSG00000000460,-0.050439,0.528678
